In [1]:
import pandas as pd
import json
import torch
import gc
from datetime import datetime
from transformers import pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma

def load_and_clean_data(input_file="nvidia_raw_intelligence.jsonl"):
    print("Step 1: Loading and Cleaning Data")
    data = []
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
            
    df = pd.DataFrame(data)
    initial_count = len(df)
    
    # Deduplicate based on URL and Title to prevent vector database bloat
    df = df.drop_duplicates(subset=['url'])
    df = df.drop_duplicates(subset=['title'])
    
    # Drop rows where summary is too short to contain semantic value
    df['summary_length'] = df['summary'].apply(lambda x: len(str(x)))
    df = df[df['summary_length'] > 20].copy()
    
    print(f"Cleaned {initial_count - len(df)} duplicate/junk records. Retained {len(df)} records.")
    return df

def calculate_sentiment(df):
    print("Step 2: Initializing Dual-Model Sentiment Pipelines...")
    # Determine device (Use GPU if available for speed, though these models are small)
    device = 0 if torch.cuda.is_available() else -1
    
    # Load Models
    finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device)
    roberta = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", device=device)
    
    def get_score(text, source):
        # Truncate text to 512 tokens to prevent transformer crashing
        short_text = str(text)[:1500] 
        
        try:
            if source == "HackerNews":
                result = roberta(short_text)[0]
            else:
                result = finbert(short_text)[0]
                
            label = result['label'].lower()
            score = result['score']
            
            # Mathematical mapping to a 0-100 scale for the dashboard
            if 'positive' in label:
                return int(50 + (score * 50))
            elif 'negative' in label:
                return int(50 - (score * 50))
            else:
                return 50 # Neutral baseline
                
        except Exception as e:
            print(f"Scoring error on text snippet: {e}")
            return 50

    print("Executing batch inference across all historical data")
    df['sentiment_score'] = df.apply(lambda row: get_score(row['summary'], row['source']), axis=1)
    
    # Brutal VRAM cleanup. We must nuke these models from memory before loading Llama 3.1
    del finbert
    del roberta
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    print("Sentiment analysis complete. VRAM purged.")
    return df

def chunk_and_vectorize(df, db_dir="./chroma_db"):
    print("Step 3: Chunking and Vector Ingestion")
    
    # Save a copy of the cleaned/scored data for forensic audit
    df.to_json("nvidia_clean_scored_intelligence.jsonl", orient="records", lines=True)
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    
    docs = []
    metadatas = []
    
    for _, row in df.iterrows():
        chunks = text_splitter.split_text(row['summary'])
        for chunk in chunks:
            docs.append(chunk)
            # Injecting critical metadata for the agent and dashboard to use later
            metadatas.append({
                "source": row['source'],
                "url": row['url'],
                "title": row['title'],
                "date": row['date'],
                "sentiment_score": row['sentiment_score']
            })
            
    print(f"Generated {len(docs)} overlapping text chunks.")
    print("Initializing BAAI/bge-base-en-v1.5 Embedding Model")
    
    embedder = HuggingFaceBgeEmbeddings(
        model_name="BAAI/bge-base-en-v1.5",
        model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
        encode_kwargs={'normalize_embeddings': True} # Forces unit vector length for cosine similarity
    )
    
    print("Writing to local ChromaDB instance")
    # This will create or update the local database directory
    vector_db = Chroma.from_texts(
        texts=docs, 
        embedding=embedder, 
        metadatas=metadatas, 
        persist_directory=db_dir
    )
    
    vector_db.persist()
    print(f"Successfully committed vectors to {db_dir}.")

if __name__ == "__main__":
    clean_df = load_and_clean_data()
    scored_df = calculate_sentiment(clean_df)
    chunk_and_vectorize(scored_df)

/home/jovyan/vault/AI_CEO_NLP/ai_ceo_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_1905/2305969702.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings


Step 1: Loading and Cleaning Data
Cleaned 46 duplicate/junk records. Retained 136 records.
Step 2: Initializing Dual-Model Sentiment Pipelines...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 39288.62it/s]


Executing batch inference across all historical data
Sentiment analysis complete. VRAM purged.
Step 3: Chunking and Vector Ingestion
Generated 141 overlapping text chunks.
Initializing BAAI/bge-base-en-v1.5 Embedding Model


/tmp/ipykernel_1905/2305969702.py:110: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7797.63it/s]


Writing to local ChromaDB instance
Successfully committed vectors to ./chroma_db.


/tmp/ipykernel_1905/2305969702.py:125: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()
